# Phase 2 — Panoplie d'algorithmes IA et moteur de benchmark

Ce notebook est prévu pour Kaggle. Il recherche automatiquement le fichier `sinert_features.csv` dans `/kaggle/input`, construit des fenêtres de 672 pas et entraîne les modèles disponibles.

Les quatre sorties prédites conjointement sont :

- AEa : énergie active ;
- AEr : énergie réactive ;
- Pmax : demande active crête ;
- Smax : demande apparente crête.

Chaque modèle produit une figure réelle/prédite sur le jeu de test. Les résultats sont sauvegardés dans `/kaggle/working/phase2_benchmark`.


In [ ]:
from pathlib import Path
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/phase2_benchmark")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if INPUT_ROOT.exists():
    candidates = sorted(INPUT_ROOT.rglob("sinert_features.csv"))
else:
    candidates = [Path("../data/processed/sinert_features.csv")]

if not candidates:
    raise FileNotFoundError(
        "Ajoutez sinert_features.csv comme dataset Kaggle dans /kaggle/input."
    )

FEATURES_PATH = candidates[0]
print("Fichier utilisé :", FEATURES_PATH)


In [ ]:
TIMESTAMP = "timestamp"
RAW_AEA = "EM_VIRTUAL kWh (kWh) [ABS]"
RAW_AER = "EM_VIRTUAL kvarh (kvarh) [ABS]"
RAW_PMAX = "EM_VIRTUAL kW dmd (kW) [MAX]"
RAW_SMAX = "EM_VIRTUAL kVA dmd (kVA) [MAX]"
CALENDAR_FEATURES = [
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
]
TARGET_LABELS = ["AEa", "AEr", "Pmax", "Smax"]

WINDOW_SIZE = 672
STEPS_PER_DAY = 96
WINDOW_STRIDE = 1
HORIZON = 1
FREQUENCY = pd.Timedelta(minutes=15)

df = pd.read_csv(FEATURES_PATH)
df[TIMESTAMP] = pd.to_datetime(df[TIMESTAMP], errors="coerce")
df = (
    df.dropna(subset=[TIMESTAMP])
    .sort_values(TIMESTAMP)
    .drop_duplicates(TIMESTAMP)
    .reset_index(drop=True)
)

raw_reference = df.copy()
df["AEa"] = df[RAW_AEA].diff()
df["AEr"] = df[RAW_AER].diff()
df.loc[df["AEa"] < 0, "AEa"] = np.nan
df.loc[df["AEr"] < 0, "AEr"] = np.nan
df["Pmax"] = df[RAW_PMAX]
df["Smax"] = df[RAW_SMAX]

ts = df[TIMESTAMP]
hour_float = ts.dt.hour + ts.dt.minute / 60
day_of_week = ts.dt.dayofweek
calendar_values = {
    "hour": hour_float,
    "day_of_week": day_of_week,
    "is_weekend": (day_of_week >= 5).astype(int),
    "hour_sin": np.sin(2 * np.pi * hour_float / 24),
    "hour_cos": np.cos(2 * np.pi * hour_float / 24),
    "day_of_week_sin": np.sin(2 * np.pi * day_of_week / 7),
    "day_of_week_cos": np.cos(2 * np.pi * day_of_week / 7),
}
for column, values in calendar_values.items():
    if column not in df.columns:
        df[column] = values

PHYSICAL_FEATURES = ["AEa", "AEr", "Pmax", "Smax"]
FEATURE_COLUMNS = PHYSICAL_FEATURES + CALENDAR_FEATURES
TARGET_COLUMNS = PHYSICAL_FEATURES
missing = [column for column in FEATURE_COLUMNS + TARGET_COLUMNS if column not in df.columns]
if missing:
    raise KeyError("Colonnes manquantes : {}".format(missing))
df = df.dropna(subset=FEATURE_COLUMNS + TARGET_COLUMNS).reset_index(drop=True)

def build_windows(frame):
    values = frame[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    targets = frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)
    timestamps = frame[TIMESTAMP].to_numpy()
    segment_start = np.r_[True, np.diff(timestamps) != FREQUENCY.to_timedelta64()]
    segment_id = np.cumsum(segment_start) - 1

    windows = []
    outputs = []
    rows = []
    for current_segment in np.unique(segment_id):
        indices = np.flatnonzero(segment_id == current_segment)
        segment_values = values[indices]
        segment_targets = targets[indices]
        segment_timestamps = timestamps[indices]
        max_start = len(indices) - WINDOW_SIZE - HORIZON + 1
        for start in range(0, max_start, WINDOW_STRIDE):
            end = start + WINDOW_SIZE
            target_index = end + HORIZON - 1
            window = segment_values[start:end]
            output = segment_targets[target_index]
            if not np.isfinite(window).all() or not np.isfinite(output).all():
                continue
            windows.append(window)
            outputs.append(output)
            rows.append({
                "window_start": pd.Timestamp(segment_timestamps[start]),
                "window_end": pd.Timestamp(segment_timestamps[end - 1]),
                "target_timestamp": pd.Timestamp(segment_timestamps[target_index]),
                "segment_id": int(current_segment),
            })

    return (
        np.stack(windows).astype(np.float32),
        np.stack(outputs).astype(np.float32),
        pd.DataFrame(rows),
    )

X, Y, metadata = build_windows(df)
print("X :", X.shape)
print("Y :", Y.shape)
print("Fenêtres créées :", len(metadata))
assert X.shape[1:] == (WINDOW_SIZE, len(FEATURE_COLUMNS))
assert Y.shape[1] == 4


In [ ]:
metadata = metadata.sort_values("target_timestamp").reset_index(drop=True)
order = metadata.index.to_numpy()
X = X[order]
Y = Y[order]

n_samples = len(Y)
train_end = int(n_samples * 0.70)
validation_end = int(n_samples * 0.85)
TRAIN = np.arange(0, train_end)
VALIDATION = np.arange(train_end, validation_end)
TEST = np.arange(validation_end, n_samples)

print({
    "train": len(TRAIN),
    "validation": len(VALIDATION),
    "test": len(TEST),
})
print("Dernière cible train :", metadata.iloc[TRAIN[-1]]["target_timestamp"])
print("Première cible test :", metadata.iloc[TEST[0]]["target_timestamp"])

X_flat = X.reshape(len(X), -1)
x_scaler = StandardScaler()
x_scaler.fit(X[TRAIN].reshape(-1, X.shape[-1]))
X_scaled = x_scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)

y_scaler = StandardScaler()
y_scaler.fit(Y[TRAIN])
Y_scaled = y_scaler.transform(Y)

def calendar_exog(timestamps):
    values = pd.Series(pd.to_datetime(timestamps))
    hour_value = values.dt.hour + values.dt.minute / 60
    weekday_value = values.dt.dayofweek
    return pd.DataFrame({
        "hour_sin": np.sin(2 * np.pi * hour_value / 24),
        "hour_cos": np.cos(2 * np.pi * hour_value / 24),
        "day_sin": np.sin(2 * np.pi * weekday_value / 7),
        "day_cos": np.cos(2 * np.pi * weekday_value / 7),
        "is_weekend": (weekday_value >= 5).astype(float),
    })

target_timestamps = metadata["target_timestamp"]
EXOG = calendar_exog(target_timestamps)
weather_tokens = ("temp", "temperature", "humidity", "irradiance", "wind", "rain")
WEATHER_COLUMNS = [
    column for column in df.columns
    if any(token in column.lower() for token in weather_tokens)
    and pd.api.types.is_numeric_dtype(df[column])
]
if WEATHER_COLUMNS:
    weather_frame = (
        df.set_index(TIMESTAMP)[WEATHER_COLUMNS]
        .groupby(level=0)
        .mean()
        .reindex(target_timestamps)
        .interpolate()
        .ffill()
        .bfill()
    )
    EXOG = pd.concat([EXOG.reset_index(drop=True), weather_frame.reset_index(drop=True)], axis=1)
print("Variables exogènes SARIMAX :", list(EXOG.columns))

prediction_store = {}
metric_rows = []
resource_rows = []

def regression_rows(model_name, split_name, y_true, y_pred):
    rows = []
    for index, label in enumerate(TARGET_LABELS):
        true = y_true[:, index]
        pred = y_pred[:, index]
        denominator = np.maximum(np.abs(true), 1e-8)
        rows.append({
            "model": model_name,
            "split": split_name,
            "target": label,
            "mae": float(np.mean(np.abs(true - pred))),
            "rmse": float(np.sqrt(np.mean((true - pred) ** 2))),
            "r2": float(1 - np.sum((true - pred) ** 2) / max(np.sum((true - true.mean()) ** 2), 1e-8)),
            "mape": float(np.mean(np.abs((true - pred) / denominator)) * 100),
        })
    return rows

def register_predictions(model_name, validation_pred, test_pred):
    prediction_store[model_name] = {
        "validation": np.asarray(validation_pred, dtype=np.float32),
        "test": np.asarray(test_pred, dtype=np.float32),
    }
    metric_rows.extend(regression_rows(model_name, "validation", Y[VALIDATION], validation_pred))
    metric_rows.extend(regression_rows(model_name, "test", Y[TEST], test_pred))

    timestamps = metadata.iloc[TEST]["target_timestamp"]
    fig, axes = plt.subplots(4, 1, figsize=(15, 13), sharex=True)
    for index, (axis, label) in enumerate(zip(axes, TARGET_LABELS)):
        axis.plot(timestamps, Y[TEST, index], label="Réel", linewidth=2)
        axis.plot(timestamps, test_pred[:, index], label="Prédit", linewidth=1.4)
        axis.set_ylabel(label)
        axis.grid(alpha=0.25)
        axis.legend(loc="upper right")
    axes[-1].set_xlabel("Date cible")
    fig.suptitle("{} — réel vs prédit sur le test".format(model_name))
    fig.tight_layout()
    safe_name = model_name.replace(" ", "_").replace("/", "_")
    fig.savefig(OUTPUT_DIR / "{}_test_reel_vs_predit.png".format(safe_name), dpi=160)
    plt.show()
    plt.close(fig)

def seasonal_from_raw(forecast_timestamps, lag_steps, fallback):
    raw = (
        df.set_index(TIMESTAMP)[TARGET_COLUMNS]
        .groupby(level=0)
        .mean()
    )
    lag = lag_steps * FREQUENCY
    result = []
    for timestamp in pd.to_datetime(forecast_timestamps):
        previous = timestamp - lag
        if previous in raw.index:
            result.append(raw.loc[previous].to_numpy(dtype=np.float32))
        else:
            result.append(fallback)
    return np.stack(result).astype(np.float32)


In [ ]:
# Modèles de référence : t-96 et t-672

train_fallback = Y[TRAIN].mean(axis=0)
naive_predictions = {}
for lag, name in [(96, "Naive_t96"), (672, "Naive_t672")]:
    validation_pred = seasonal_from_raw(
        metadata.iloc[VALIDATION]["target_timestamp"],
        lag,
        train_fallback,
    )
    test_pred = seasonal_from_raw(
        metadata.iloc[TEST]["target_timestamp"],
        lag,
        Y[np.r_[TRAIN, VALIDATION]].mean(axis=0),
    )
    naive_predictions[name] = (validation_pred, test_pred)
    register_predictions(name, validation_pred, test_pred)

# SARIMAX avec variables calendaires et météo lorsqu'elles existent.
def sarimax_forecast(fit_indices, forecast_indices):
    predictions = []
    for target_index in range(Y.shape[1]):
        model = SARIMAX(
            Y[fit_indices, target_index],
            exog=EXOG.iloc[fit_indices],
            order=(1, 0, 1),
            seasonal_order=(1, 0, 1, 7),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fitted = model.fit(disp=False, maxiter=100)
        forecast = fitted.get_forecast(
            steps=len(forecast_indices),
            exog=EXOG.iloc[forecast_indices],
        ).predicted_mean
        predictions.append(np.asarray(forecast, dtype=np.float32))
    return np.column_stack(predictions)

sarimax_validation = sarimax_forecast(TRAIN, VALIDATION)
sarimax_test = sarimax_forecast(np.r_[TRAIN, VALIDATION], TEST)
register_predictions("SARIMAX", sarimax_validation, sarimax_test)

# Modèles Machine Learning sur la représentation aplatie 672 × 11.
def fit_sklearn_model(name, estimator):
    start = time.perf_counter()
    estimator.fit(X_scaled[TRAIN].reshape(len(TRAIN), -1), Y_scaled[TRAIN])
    validation_pred = y_scaler.inverse_transform(estimator.predict(
        X_scaled[VALIDATION].reshape(len(VALIDATION), -1)
    ))
    test_pred = y_scaler.inverse_transform(estimator.predict(
        X_scaled[TEST].reshape(len(TEST), -1)
    ))
    elapsed = time.perf_counter() - start
    resource_rows.append({"model": name, "training_time_seconds": elapsed, "parameter_count": np.nan, "parameter_source": "not_available_for_sklearn_export"})
    print("{} entraîné en {:.1f} s".format(name, elapsed))
    register_predictions(name, validation_pred, test_pred)

fit_sklearn_model(
    "Random_Forest",
    RandomForestRegressor(
        n_estimators=250,
        random_state=SEED,
        n_jobs=-1,
        max_features="sqrt",
    ),
)

fit_sklearn_model(
    "SVR",
    MultiOutputRegressor(
        SVR(C=10.0, epsilon=0.05, gamma="scale")
    ),
)

try:
    from xgboost import XGBRegressor
    fit_sklearn_model(
        "XGBoost",
        MultiOutputRegressor(
            XGBRegressor(
                n_estimators=300,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                random_state=SEED,
                n_jobs=-1,
            )
        ),
    )
except ImportError:
    print("XGBoost indisponible : modèle ignoré.")

try:
    from lightgbm import LGBMRegressor
    fit_sklearn_model(
        "LightGBM",
        MultiOutputRegressor(
            LGBMRegressor(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                random_state=SEED,
                verbosity=-1,
            )
        ),
    )
except ImportError:
    print("LightGBM indisponible : modèle ignoré.")


In [ ]:
# Réseaux récurrents, convolutifs, résiduels et Transformer

RUN_DEEP_LEARNING = True
EPOCHS = 25
BATCH_SIZE = 32


try:
    import tensorflow as tf
    from tensorflow.keras import Model
    from tensorflow.keras import callbacks, layers
    tf.keras.utils.set_random_seed(SEED)
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow indisponible : les modèles Deep Learning sont ignorés.")

def residual_block(inputs, filters, dilation):
    shortcut = inputs
    hidden = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation)(inputs)
    hidden = layers.BatchNormalization()(hidden)
    hidden = layers.Activation("relu")(hidden)
    hidden = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation)(hidden)
    hidden = layers.BatchNormalization()(hidden)
    if inputs.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)
    return layers.Activation("relu")(layers.Add()([shortcut, hidden]))

def build_deep_model(name):
    inputs = layers.Input(shape=(WINDOW_SIZE, len(FEATURE_COLUMNS)))
    if name == "LSTM":
        hidden = layers.LSTM(64)(inputs)
    elif name == "BiLSTM":
        hidden = layers.Bidirectional(layers.LSTM(32))(inputs)
    elif name == "GRU":
        hidden = layers.GRU(64)(inputs)
    elif name == "CNN1D":
        hidden = layers.Conv1D(64, 7, padding="same", activation="relu")(inputs)
        hidden = layers.MaxPooling1D(4)(hidden)
        hidden = layers.Conv1D(64, 5, padding="same", activation="relu")(hidden)
        hidden = layers.GlobalAveragePooling1D()(hidden)
    elif name == "ResNet1D_TCN":
        hidden = layers.Conv1D(64, 1, padding="same")(inputs)
        for dilation in (1, 2, 4, 8):
            hidden = residual_block(hidden, 64, dilation)
        hidden = layers.GlobalAveragePooling1D()(hidden)
    elif name == "Transformer":
        hidden = layers.Dense(64)(inputs)
        attention = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1)(
            hidden, hidden
        )
        hidden = layers.LayerNormalization()(hidden + attention)
        feed_forward = layers.Dense(128, activation="relu")(hidden)
        feed_forward = layers.Dense(64)(feed_forward)
        hidden = layers.LayerNormalization()(hidden + feed_forward)
        hidden = layers.GlobalAveragePooling1D()(hidden)
    elif name == "CNN_LSTM":
        hidden = layers.Conv1D(64, 5, padding="same", activation="relu")(inputs)
        hidden = layers.MaxPooling1D(4)(hidden)
        hidden = layers.LSTM(64)(hidden)
    else:
        raise ValueError("Modèle Deep Learning inconnu : {}".format(name))

    hidden = layers.Dense(64, activation="relu")(hidden)
    hidden = layers.Dropout(0.2)(hidden)
    outputs = layers.Dense(4, name="multi_output_regression")(hidden)
    return Model(inputs, outputs, name=name)

if RUN_DEEP_LEARNING and TENSORFLOW_AVAILABLE:
    for model_name in [
        "LSTM",
        "BiLSTM",
        "GRU",
        "CNN1D",
        "ResNet1D_TCN",
        "Transformer",
        "CNN_LSTM",
    ]:
        print("Entraînement :", model_name)
        model = build_deep_model(model_name)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss="mse",
            metrics=["mae"],
        )
        stop = callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
        start = time.perf_counter()
        model.fit(
            X_scaled[TRAIN],
            Y_scaled[TRAIN],
            validation_data=(X_scaled[VALIDATION], Y_scaled[VALIDATION]),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[stop],
            verbose=1,
        )
        elapsed = time.perf_counter() - start
        resource_rows.append({"model": model_name, "training_time_seconds": elapsed, "parameter_count": int(model.count_params()), "parameter_source": "tensorflow_model_count_params"})
        validation_pred = y_scaler.inverse_transform(
            model.predict(X_scaled[VALIDATION], verbose=0)
        )
        test_pred = y_scaler.inverse_transform(
            model.predict(X_scaled[TEST], verbose=0)
        )
        print("{} entraîné en {:.1f} s".format(model_name, elapsed))
        register_predictions(model_name, validation_pred, test_pred)
        model.save(OUTPUT_DIR / "{}.keras".format(model_name))


In [ ]:
metrics = pd.DataFrame(metric_rows)
metrics = metrics.sort_values(["split", "target", "rmse"])
metrics.to_csv(OUTPUT_DIR / "benchmark_metrics_all_models.csv", index=False)

prediction_rows = []
for model_name, split_data in prediction_store.items():
    for split_name, split_indices in [("validation", VALIDATION), ("test", TEST)]:
        prediction = split_data[split_name]
        base = pd.DataFrame({
            "model": model_name,
            "split": split_name,
            "target_timestamp": metadata.iloc[split_indices]["target_timestamp"].to_numpy(),
        })
        for index, label in enumerate(TARGET_LABELS):
            base["real_" + label] = Y[split_indices, index]
            base["pred_" + label] = prediction[:, index]
        prediction_rows.append(base)

predictions = pd.concat(prediction_rows, ignore_index=True)
predictions.to_csv(OUTPUT_DIR / "benchmark_predictions_all_models.csv", index=False)
pd.DataFrame(resource_rows).to_csv(OUTPUT_DIR / "benchmark_resource_usage.csv", index=False)

raw_reference_indexed = raw_reference.set_index(TIMESTAMP)
test_timestamps = metadata.iloc[TEST]["target_timestamp"]
previous_timestamps = test_timestamps - pd.Timedelta(minutes=15)
previous_values = raw_reference_indexed.reindex(previous_timestamps)
base_ae = previous_values[[RAW_AEA, RAW_AER]].to_numpy()

for model_name, split_data in prediction_store.items():
    predicted_increment = split_data["test"][:, :2]
    real_increment = Y[TEST, :2]
    real_cumulative = base_ae + real_increment
    predicted_cumulative = base_ae + predicted_increment

    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    for index, (axis, label, unit) in enumerate([
        (axes[0], "AEa cumulée", "kWh"),
        (axes[1], "AEr cumulée", "kVARh"),
    ]):
        axis.plot(test_timestamps, real_cumulative[:, index], label="Réel", linewidth=2)
        axis.plot(test_timestamps, predicted_cumulative[:, index], label="Prédit")
        axis.set_title("{} — {} réel vs prédit".format(model_name, label))
        axis.set_ylabel(unit)
        axis.legend()
        axis.grid(alpha=0.25)
    axes[-1].set_xlabel("Date cible")
    fig.tight_layout()
    safe_name = model_name.replace(" ", "_").replace("/", "_")
    fig.savefig(OUTPUT_DIR / "{}_cumul_reel_vs_predit.png".format(safe_name), dpi=160)
    plt.show()
    plt.close(fig)

configuration = {
    "features_path": str(FEATURES_PATH),
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "target_labels": TARGET_LABELS,
    "window_size": WINDOW_SIZE,
    "window_stride": WINDOW_STRIDE,
    "horizon": HORIZON,
    "split_sizes": {
        "train": len(TRAIN),
        "validation": len(VALIDATION),
        "test": len(TEST),
    },
    "weather_columns": WEATHER_COLUMNS,
    "deep_learning_enabled": RUN_DEEP_LEARNING and TENSORFLOW_AVAILABLE,
    "models": sorted(prediction_store),
}
(OUTPUT_DIR / "benchmark_configuration.json").write_text(
    json.dumps(configuration, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(metrics[metrics["split"] == "test"].sort_values("rmse").to_string(index=False))
print("Résultats :", OUTPUT_DIR)
